## Model 1 — Binárna detekcia poruchy

Model 1 vykonáva binárnu klasifikáciu 6-hodinových blokov na normálnu prevádzku a poruchu. 
Používa algoritmus XGBoost optimalizovaný pomocou Optuna (150 pokusov, TPE sampler).
Minimálne požiadavky: recall ≥ 0.62, precision ≥ 0.60.
Natrénovaný model sa ukladá do súboru `models/model1.pkl`.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from xgboost import XGBClassifier
from sklearn.metrics import (
    recall_score, precision_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_sample_weight
from collections import defaultdict
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH   = 'data/final_typy.csv'
MODELS_DIR  = 'models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Načítanie a zoradenie dát
df = pd.read_csv(DATA_PATH)
df['t_utc'] = pd.to_datetime(df['t_utc'])
df_sorted = df.sort_values(['eic', 't_utc']).reset_index(drop=True)

fault_cols = [c for c in df_sorted.columns if c.startswith('fault_type')]

# Rozdelenie podľa EIC
eic_fault_profile = {}
for eic in sorted(df_sorted['eic'].unique()):
    eic_data = df_sorted[df_sorted['eic'] == eic]
    profile  = tuple(int(eic_data[col].sum() > 0) for col in sorted(fault_cols))
    eic_fault_profile[eic] = profile

profile_groups = defaultdict(list)
for eic, profile in eic_fault_profile.items():
    profile_groups[profile].append(eic)

np.random.seed(42)
train_eics, test_eics = [], []

for profile, eics in sorted(profile_groups.items()):
    eics_shuffled = eics.copy()
    np.random.shuffle(eics_shuffled)
    if len(eics_shuffled) == 1:
        train_eics.extend(eics_shuffled)
    elif len(eics_shuffled) == 2:
        train_eics.append(eics_shuffled[0])
        test_eics.append(eics_shuffled[1])
    else:
        split_n = max(1, int(len(eics_shuffled) * 0.8))
        train_eics.extend(eics_shuffled[:split_n])
        test_eics.extend(eics_shuffled[split_n:])

train_eics = set(train_eics)
test_eics  = set(test_eics)
assert len(train_eics & test_eics) == 0, "Existuju spolocne EIC v train a test!"

train_df = df_sorted[df_sorted['eic'].isin(train_eics)].copy()
test_df  = df_sorted[df_sorted['eic'].isin(test_eics)].copy()

print(f"Trénovacia množina — EIC: {len(train_eics)} | Záznamy: {len(train_df)} | Poruchy: {train_df['porucha?'].mean()*100:.1f}%")
print(f"Testovacia množina — EIC: {len(test_eics)}  | Záznamy: {len(test_df)}  | Poruchy: {test_df['porucha?'].mean()*100:.1f}%")
print(f"Prienik EIC: {len(train_eics & test_eics)}")

# Definícia príznakov
feature_cols = [
    'i1_norm_max', 'i1_norm_mean', 'i1_norm_min',
    'i2_norm_max', 'i2_norm_mean', 'i2_norm_min',
    'i3_norm_max', 'i3_norm_mean', 'i3_norm_min',
    'u1_norm_min', 'u1_norm_max', 'u1_norm_mean',
    'u2_norm_min', 'u2_norm_max', 'u2_norm_mean',
    'u3_norm_min', 'u3_norm_max', 'u3_norm_mean',
    'u1_norm_p2p', 'u1_norm_std', 'u1_norm_mean_abs_diff',
    'u2_norm_p2p', 'u2_norm_mean_abs_diff',
    'u3_norm_p2p', 'u3_norm_mean_abs_diff',
    'u1_norm_kurtosis',
]

missing = [c for c in feature_cols if c not in df_sorted.columns]
if missing:
    raise ValueError(f"Chýbajúce príznaky: {missing}")

train_medians = train_df[feature_cols].median()

X_train = train_df[feature_cols].fillna(train_medians)
y_train = train_df['porucha?'].astype(int)

X_test  = test_df[feature_cols].fillna(train_medians)
y_test  = test_df['porucha?'].astype(int)

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())

if pos == 0 or neg == 0:
    raise ValueError("Trénovacia množina obsahuje iba jednu triedu.")

print(f"\nTrénovacia množina: {len(X_train)} záznamov | Normálna prevádzka: {neg} | Porucha: {pos} | Pomer: {neg/pos:.2f}")
print(f"Testovacia množina: {len(X_test)} záznamov  | Podiel porúch: {y_test.mean()*100:.1f}%")

total    = len(X_train) + len(X_test)
test_pct = len(X_test) / total * 100
if test_pct < 40:
    print(f"Testovacia množina tvorí {test_pct:.1f}% — odporúča sa rozdelenie 60/40")
else:
    print(f"Testovacia množina tvorí {test_pct:.1f}%")

# Váhy vzoriek pre vyrovnanie tried
sample_weight_train = compute_sample_weight(class_weight='balanced', y=y_train)

# Minimálne požiadavky na kvalitu modelu
MIN_RECALL    = 0.62
MIN_PRECISION = 0.60

# Optimalizácia hyperparametrov pomocou Optuna
def objective(trial):
    params = {
        'n_estimators':     trial.suggest_int('n_estimators', 150, 600),
        'max_depth':        trial.suggest_int('max_depth', 3, 8),
        'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.15),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'subsample':        trial.suggest_float('subsample', 0.65, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.65, 1.0),
        'gamma':            trial.suggest_float('gamma', 0, 4),
        'reg_alpha':        trial.suggest_float('reg_alpha', 0, 2),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1, 8),
        'random_state':     42,
        'eval_metric':      'logloss',
        'tree_method':      'hist',
        'n_jobs':           -1,
    }

    model = XGBClassifier(**params)
    model.fit(X_train, y_train, sample_weight=sample_weight_train)
    y_prob = model.predict_proba(X_test)[:, 1]

    best_local_score = 0.0
    for thresh in np.arange(0.25, 0.76, 0.05):
        y_pred = (y_prob >= thresh).astype(int)
        r = recall_score(y_test, y_pred, zero_division=0)
        p = precision_score(y_test, y_pred, zero_division=0)

        if r < MIN_RECALL or p < MIN_PRECISION:
            continue

        beta       = 0.8
        fbeta      = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        fp_rate    = ((y_test == 0) & (y_pred == 1)).sum() / max((y_test == 0).sum(), 1)
        fp_penalty = max(0.0, fp_rate - 0.15)
        score      = fbeta - 0.5 * fp_penalty

        if score > best_local_score:
            best_local_score = score

    return best_local_score

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=150, show_progress_bar=True)

print(f"\nNajlepsie skore Optuna: {study.best_value:.4f}")
for k, v in study.best_params.items():
    print(f"   {k}: {v}")

# Trénovanie finálneho modelu
model1 = XGBClassifier(
    **study.best_params,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    n_jobs=-1
)
model1.fit(X_train, y_train, sample_weight=sample_weight_train)

# Výber optimálnej prahovej hodnoty
y_prob_opt = model1.predict_proba(X_test)[:, 1]

print("\nVýber prahovej hodnoty — Model 1:")
print(f"{'Prah':<8} {'Návratnosť':<12} {'Presnosť':<12} {'F1':<8} {'FPR':<8} {'Vynechané':<12} {'FP'}")
print("-" * 72)

best_thresh = 0.5
best_score  = -999.0

for thresh in np.arange(0.20, 0.81, 0.05):
    y_p    = (y_prob_opt >= thresh).astype(int)
    r      = recall_score(y_test, y_p, zero_division=0)
    p      = precision_score(y_test, y_p, zero_division=0)
    f1     = f1_score(y_test, y_p, zero_division=0)
    missed = int(((y_test == 1) & (y_p == 0)).sum())
    fp     = int(((y_test == 0) & (y_p == 1)).sum())
    fpr    = fp / max((y_test == 0).sum(), 1)

    if r < MIN_RECALL or p < MIN_PRECISION:
        mark  = " [preskocene]"
        score = -999.0
    else:
        beta       = 0.8
        fbeta      = (1 + beta**2) * p * r / (beta**2 * p + r + 1e-9)
        fp_penalty = max(0.0, fpr - 0.15)
        score      = fbeta - 0.5 * fp_penalty
        mark       = " <-" if score > best_score else ""

    if score > best_score:
        best_score  = score
        best_thresh = round(thresh, 2)

    print(f"{thresh:<8.2f} {r:<12.3f} {p:<12.3f} {f1:<8.3f} {fpr:<8.3f} {missed:<12} {fp}{mark}")

# Finálne hodnotenie modelu
THRESHOLD    = best_thresh
y_pred_final = (y_prob_opt >= THRESHOLD).astype(int)

print(f"\nZvolená prahová hodnota Model 1: {THRESHOLD}")
print(classification_report(
    y_test, y_pred_final,
    target_names=['Normálna prevádzka', 'Porucha'],
    zero_division=0
))

cm        = confusion_matrix(y_test, y_pred_final)
fp_final  = cm[0, 1]
fpr_final = fp_final / max(cm[0].sum(), 1)

print(f"  Správne klasifikovaná normálna prevádzka: {cm[0,0]}")
print(f"  Falošné poplachy (FP):                    {fp_final}  (FPR = {fpr_final:.1%})")
print(f"  Nezachytené poruchy (FN):                 {cm[1,0]}")
print(f"  Správne zachytené poruchy (TP):           {cm[1,1]}")
print(f"\nModel 1 finalizovaný s prahovou hodnotou {THRESHOLD}")

# Uloženie modelu do priečinka models
joblib.dump(model1, f'{MODELS_DIR}/model1.pkl')
print(f"Uložené: {MODELS_DIR}/model1.pkl")

# Detekčná funkcia pre nasadenie
feature_cols_model1  = feature_cols
train_medians_model1 = train_medians

def detect_fault(X_new):
    X_new = X_new[feature_cols_model1].fillna(train_medians_model1)
    prob  = model1.predict_proba(X_new)[:, 1]
    pred  = (prob >= THRESHOLD).astype(int)
    return pred, prob

In [7]:
## 

Trénovacia množina — EIC: 78 | Záznamy: 153980 | Poruchy: 64.4%
Testovacia množina — EIC: 28  | Záznamy: 54378  | Poruchy: 65.5%
Prienik EIC: 0

Trénovacia množina: 153980 záznamov | Normálna prevádzka: 54849 | Porucha: 99131 | Pomer: 0.55
Testovacia množina: 54378 záznamov  | Podiel porúch: 65.5%
Testovacia množina tvorí 26.1% — odporúča sa rozdelenie 60/40
Error displaying widget

Najlepsie skore Optuna: 0.8348
   n_estimators: 372
   max_depth: 3
   learning_rate: 0.03654441157249876
   min_child_weight: 10
   subsample: 0.8998117100964699
   colsample_bytree: 0.9128992120030185
   gamma: 1.3652321445664186
   reg_alpha: 0.2987789821798761
   reg_lambda: 4.123010889087539

Výber prahovej hodnoty — Model 1:
Prah     Návratnosť   Presnosť     F1       FPR      Vynechané    FP
------------------------------------------------------------------------
0.20     0.919        0.789        0.849    0.467    2876         8749 <-
0.25     0.906        0.814        0.858    0.394    3339         7374 <-
0.30     0.898        0.843        0.869    0.319    3649         5973 <-
0.35     0.890        0.851        0.870    0.297    3923         5560 <-
0.40     0.878        0.852        0.864    0.291    4364         5444
0.45     0.870        0.855        0.862    0.282    4620         5275 <-
0.50     0.864        0.860        0.862    0.268    4862         5024 <-
0.55     0.860        0.864        0.862    0.257    4981         4813 <-
0.60     0.857        0.869        0.863    0.247    5085         4622 <-
0.65     0.853        0.873        0.863    0.236    5255         4427 <-
0.70     0.833        0.884        0.858    0.208    5942         3891 <-
0.75     0.734        0.892        0.805    0.169    9488         3169
0.80     0.659        0.896        0.759    0.146    12153        2739

Zvolená prahová hodnota Model 1: 0.7
                    precision    recall  f1-score   support

Normálna prevádzka       0.71      0.79      0.75     18736
           Porucha       0.88      0.83      0.86     35642

          accuracy                           0.82     54378
         macro avg       0.80      0.81      0.80     54378
      weighted avg       0.83      0.82      0.82     54378

  Správne klasifikovaná normálna prevádzka: 14845
  Falošné poplachy (FP):                    3891  (FPR = 20.8%)
  Nezachytené poruchy (FN):                 5942
  Správne zachytené poruchy (TP):           29700

Model 1 finalizovaný s prahovou hodnotou 0.7
Uložené: models/model1.pkl